# 00 — One-time Setup

Run this notebook **once** (per Google account + Drive folder) before using the `01_run_mlp3.ipynb` or `02_run_mlp3_fixed.ipynb` runners.

**What it does:**
1. Mounts Google Drive (expected account: `hert7739@ox.ac.uk`)
2. Creates `results_pcqm4m_subset/` and `datasets/` folders on Drive (if missing)
3. Clones the GitHub repo into the VM
4. Installs dependencies
5. Symlinks `results_pcqm4m_subset/` + `datasets/` in the repo to those Drive folders
6. Logs into WandB
7. **Dataset step:** if `MyDrive/datasets/pcqm4m-v2/processed/` already contains a large OGB `.pt` file (e.g. you uploaded your `datasets/` tree from another machine), that step is **skipped**. Otherwise OGB downloads + processes PCQM4Mv2 onto Drive (~30 min).

**Why Drive?** The processed PCQM4Mv2 tensor is ~7–8 GB. Keeping it on Drive means every new Colab VM only remounts and symlinks — you do not re-download.

**Runtime:** GPU is not strictly needed for setup, but use GPU anyway so the dataset loader sanity-check verifies CUDA works when you run a full OGB load.

## 1. Configuration

In [ ]:
# =========================================================
# EDIT IF NEEDED
# =========================================================
GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"   # where results_pcqm4m_subset/ + datasets/ will live
# If True, always run OGB download/process (slow). If False and a large processed .pt already
# exists under MyDrive/datasets/pcqm4m-v2/processed/, the dataset cell skips the heavy work.
FORCE_DOWNLOAD_PCQM4M = False
# When reusing Drive data, set True to call PygPCQM4Mv2Dataset anyway (verifies splits; uses RAM/time).
VERIFY_AFTER_REUSE = False
# =========================================================

DRIVE_RESULTS  = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_DATASETS = f"{DRIVE_WORKSPACE}/datasets"
print("Drive results dir: ", DRIVE_RESULTS)
print("Drive datasets dir:", DRIVE_DATASETS)

## 2. Mount Google Drive

Sign in with **hert7739@ox.ac.uk** when the popup appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Create Drive folders

In [ ]:
import os
os.makedirs(DRIVE_RESULTS,  exist_ok=True)
os.makedirs(DRIVE_DATASETS, exist_ok=True)
!ls -la {DRIVE_WORKSPACE} | head -20

## 4. Clone the repo

In [ ]:
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
%cd gdl__2
!pwd && git log -1 --oneline

## 5. Install dependencies (~3 min)

In [ ]:
!bash run_colab/setup.sh

## 6. Symlink `results_pcqm4m_subset/` and `datasets/` to Drive

After this, anything written into these dirs by the training code lands directly on Drive.

In [ ]:
!rm -rf results_pcqm4m_subset datasets
!ln -s {DRIVE_RESULTS}  results_pcqm4m_subset
!ln -s {DRIVE_DATASETS} datasets
!ls -la results_pcqm4m_subset datasets

## 7. Log in to WandB

API key at [wandb.ai/authorize](https://wandb.ai/authorize).

In [ ]:
import wandb
wandb.login()

## 8. PCQM4Mv2 on Drive (download **or** reuse)

- **First time / empty Drive folder:** OGB downloads `data.csv.gz` (~52 MB) and builds a large processed `.pt` (~7–8 GB) **on Drive** via the `datasets` symlink (~30 min).
- **You already have `MyDrive/datasets/pcqm4m-v2/`** (uploaded or from a previous run): with `FORCE_DOWNLOAD_PCQM4M = False`, this cell **skips** the download and only lists what it found. Set `VERIFY_AFTER_REUSE = True` if you want OGB to open the dataset and print split sizes (slower, more RAM).

**Coffee break** only if you see the download path kick in.

In [ ]:
from pathlib import Path

def _pcqm4m_processed_looks_ready():
    """True if a large OGB-style processed tensor is already under datasets/ (symlink → Drive)."""
    proc = Path("datasets") / "pcqm4m-v2" / "processed"
    if not proc.is_dir():
        return False
    for p in proc.glob("*.pt"):
        if p.is_file() and p.stat().st_size > 500_000_000:  # >500 MB → almost certainly full processed data
            return True
    return False

reuse = _pcqm4m_processed_looks_ready() and not FORCE_DOWNLOAD_PCQM4M

if reuse:
    proc = Path("datasets") / "pcqm4m-v2" / "processed"
    print("Reusing existing PCQM4Mv2 under datasets/pcqm4m-v2/processed/ (Drive-backed).")
    for p in sorted(proc.glob("*.pt")):
        if p.is_file():
            print(f"  {p.name}  {p.stat().st_size / 1e9:.2f} GB")
    if VERIFY_AFTER_REUSE:
        print("VERIFY_AFTER_REUSE=True → loading via OGB...")
        from ogb.lsc import PygPCQM4Mv2Dataset
        ds = PygPCQM4Mv2Dataset(root="datasets")
        print("Dataset ready:", len(ds), "graphs")
        print({k: len(v) for k, v in ds.get_idx_split().items()})
    else:
        print("Skipping PygPCQM4Mv2Dataset() for a fast setup. Training notebooks will load data when needed.")
else:
    if FORCE_DOWNLOAD_PCQM4M and _pcqm4m_processed_looks_ready():
        print("FORCE_DOWNLOAD_PCQM4M=True — re-running OGB (may refresh / rebuild as needed)...")
    else:
        print("No large processed .pt found — downloading + processing (~30 min)...")
    from ogb.lsc import PygPCQM4Mv2Dataset
    ds = PygPCQM4Mv2Dataset(root="datasets")
    print("Dataset ready:", len(ds), "graphs")
    print({k: len(v) for k, v in ds.get_idx_split().items()})

## 9. Verify Drive contents

In [ ]:
!du -sh {DRIVE_DATASETS}/pcqm4m-v2/processed/* 2>/dev/null
!ls -la {DRIVE_RESULTS}

## ✅ Done

Now open one of:
- **`01_run_mlp3.ipynb`** — trains L-HKS K=16 MLP3 (learned diffusion times)
- **`02_run_mlp3_fixed.ipynb`** — trains L-HKS K=16 MLP3 (frozen diffusion times)

Each of those will mount the same Drive and re-use the dataset you just processed.